<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/Final_OT_Cybersecurity_Risk_Assessment_03_January_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
from google.colab import userdata
from openai import OpenAI

# ---------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    if not os.path.exists(path):
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):

    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):

    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI 08-09_V6.pdf"

    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    R, R_expl = risk_estimator(L, I, heatmap_path)

    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    run_btn = gr.Button("Run Assessment")

    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://c1d09f6deb544410b8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [6]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    """
    Multi-factor likelihood agent:
    - Threat actor capability
    - Vulnerability exploitability (LLM)
    - Exposure
    - Historical occurrences (LLM)
    Final likelihood = max of the four factors.
    """
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    # Driving factor = highest severity
    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    # Explanation generation
    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://2282544a90e7c01730.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [7]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Modified to use public web APIs for Vulnerability Exploitability & Exposure)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (with Web Data)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD public API (no auth needed for low volume).
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org free API.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent (now web-informed):
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (LLM)
    Final likelihood = max of the four factors.
    """
    tac = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    # Defaults before enrichment
    vuln = None
    exp_factor = None

    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS
        epss = fetch_epss(cve)
        if epss is not None:
            exp_factor = map_epss_to_likelihood(epss)

    # Fallbacks if web data is not available
    if vuln is None:
        vuln = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_factor is None:
        exp_factor = clamp_likelihood(exposure_input)

    # Historical occurrences still via LLM
    hist = infer_likelihood_from_model(title + " (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp_factor],
        "historical_occurrences": L2S[hist]
    }

    # Driving factor = highest severity
    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    # Explanation generation
    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp_factor}, HIST={hist}.
Driving factor: {max_factor}.
If a CVE was available, mention how public exploitability and EPSS data influenced the assessment.
CVE used (if any): {cve}
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp_factor,
        "historical_occurrences": hist,
        "cve": cve
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (now web-informed for exploitability & exposure if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://fc9deb8f9c1fcc384c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [11]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + API-key integration)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# >>> CHANGED: Load external API keys from Colab userdata
# You should define these keys in Colab:
# userdata["nvd_api_key"], userdata["epss_api_key"], userdata["history_api_key"]
NVD_API_KEY = userdata.get("nvd_api_key")
EPSS_API_KEY = userdata.get("epss_api_key")          # optional; included per your request
HISTORY_API_KEY = userdata.get("history_api_key")    # for historical occurrence API

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Uses API key if available (if your plan requires it).
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    headers = {}
    params = {}

    # Some deployments might use API keys via headers or params.
    # Adjust according to your EPSS account requirements.
    if EPSS_API_KEY:
        headers["Authorization"] = f"Bearer {EPSS_API_KEY}"

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

# >>> CHANGED: Historical occurrence via external API

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    You can adapt URL/structure to your real service.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        # Expecting a float field "likelihood" between 0 and 1
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

# >>> CHANGED: New weighted + override likelihood_agent

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    # >>> CHANGED: likelihood_agent now returns (label, numeric, explanation, details)
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


SecretNotFoundError: Secret epss_api_key does not exist.

In [13]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + API-key integration, free EPSS)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    You can adapt URL/structure to your real service.
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        # Expecting a float field "likelihood" between 0 and 1
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://04ac62b20b84832287.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [15]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Weighted + RAG for Compliance)
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use ONLY the context below and your knowledge of regulatory principles:

{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations
- potential for regulatory findings or violations
- reporting obligations or enforcement actions

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Consider:
- public perception
- media attention
- stakeholder and investor confidence

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model:

    Factors (labels on 0–4 scale):
    - Safety
    - Availability
    - Confidentiality
    - Integrity
    - Compliance (RAG-based from FANR-REG-08 or similar)
    - Reputation (LLM-based)

    Weights (sum to 1.0, tuned for nuclear OT):
    - Safety:         0.35
    - Availability:   0.25
    - Integrity:      0.15
    - Confidentiality:0.10
    - Compliance:     0.10
    - Reputation:     0.05

    Conservative overrides:
    - If Safety >= major   → final impact >= major
    - If Compliance = severe → final impact >= major
    - If asset criticality = Very High → bump one level (capped at severe)
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
Highlight how the matrix embodies the organization's risk appetite and
the interaction between likelihood and impact categories.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
Prioritize safety, defense-in-depth, and regulatory alignment for nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
Emphasize nuclear OT considerations, regulatory expectations, and
how the controls reduce likelihood and/or impact.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical JSON):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical JSON):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
Include:
- Executive summary
- Risk description
- Likelihood analysis
- Impact analysis
- Overall risk rating
- Recommended controls and implementation priorities
- Notes for regulators and auditors
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://a4a321e683618821e5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [16]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# OT-NUCLEAR-FOCUSED PROMPTS AND CONTROLS
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# --------------------------------------------------
# Global OT-nuclear system prompt for all chat calls
# --------------------------------------------------

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain your reasoning in 1–2 short sentences in a nuclear OT context, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Explicitly reference OT realities such as legacy ICS/SCADA, limited patching windows,
  deterministic protocols, and physical process coupling where relevant.
Avoid generic IT-centric language.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Weighted + RAG for Compliance)
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model:

    Factors (labels on 0–4 scale):
    - Safety
    - Availability
    - Confidentiality
    - Integrity
    - Compliance (RAG-based from FANR-REG-08 or similar)
    - Reputation (LLM-based)

    Weights (sum to 1.0, tuned for nuclear OT):
    - Safety:         0.35
    - Availability:   0.25
    - Integrity:      0.15
    - Confidentiality:0.10
    - Compliance:     0.10
    - Reputation:     0.05

    Conservative overrides:
    - If Safety >= major        → final impact >= major
    - If Compliance = severe    → final impact >= major
    - If asset criticality = Very High → bump one level (capped at severe)
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation – OT-nuclear framing
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    # Strong OT-nuclear-focused RAG query
    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator (OT-nuclear framing)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset category = {asset_category}
Asset criticality = {asset_criticality}
Risk title = {title}
Causes = {causes}

Likelihood = {likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical JSON):
{json.dumps(likelihood_details, indent=2)}

Impact = {impact}
Impact basis:
{impact_basis}

Impact details (technical JSON):
{json.dumps(impact_details, indent=2)}

Risk rating = {rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Controls rationale:
{controls_rationale}

Report requirements:
- Write professionally, as if for a nuclear OT cyber risk committee.
- Use headings and bullet points.
- Sections:
  1) Executive Summary (in nuclear OT terms, not generic IT)
  2) Risk Description (including asset, process, and safety relevance)
  3) Likelihood Analysis (highlight OT-specific drivers)
  4) Impact Analysis (safety, regulatory, operational, reputation)
  5) Overall Risk Rating (link to nuclear risk appetite)
  6) Recommended Controls and Implementation Priorities
  7) Notes for Regulators and Auditors (FANR/NEI/NRC alignment)

- Emphasize:
  • physical process and safety implications,
  • operational constraints (maintenance windows, legacy systems),
  • regulatory expectations,
  • why generic IT practices are insufficient without OT-specific adaptation.

Avoid cloud-centric or purely IT-enterprise language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations (OT-nuclear-focused)
    5. Final report generation (OT-nuclear framing)
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls (OT-nuclear-focused)
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report (OT-nuclear framing)
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full OT-nuclear-focused multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://64231a597c43250071.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [17]:
# ============================================================
# OT Nuclear Risk Assessment – Refactored, Config-Driven, OT-Nuclear Focused
# Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import time
import hashlib
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ============================================================
# 1) CONFIGURATION & CONSTANTS
# ============================================================

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Central risk configuration (weights, thresholds, paths, etc.)
RISK_CONFIG = {
    "likelihood_weights": {
        "tac": 0.25,
        "vuln": 0.35,
        "exp": 0.25,
        "hist": 0.15,
    },
    "impact_weights": {
        "safety": 0.35,
        "availability": 0.25,
        "integrity": 0.15,
        "confidentiality": 0.10,
        "compliance": 0.10,
        "reputation": 0.05,
    },
    "epss_breakpoints": [0.05, 0.20, 0.50, 0.75],
    "history_breakpoints": [0.05, 0.20, 0.50, 0.75],
    "paths": {
        "compliance_pdf": "data/FANR-REG-08_V2.pdf",
        "heatmap": "data/Nuclear_OT_Risk_Heatmap.xlsx",
        "control_library": "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx",
    },
}

# Optional asset-specific tweaks (example – currently small)
ASSET_PROFILES = {
    "Operator Workstation": {"safety_weight_bonus": 0.00, "integrity_weight_bonus": 0.00},
    "Engineering Workstation": {"safety_weight_bonus": 0.00, "integrity_weight_bonus": 0.05},
    "Server": {"safety_weight_bonus": 0.00, "integrity_weight_bonus": 0.00},
    "Network Switch": {"safety_weight_bonus": 0.00, "integrity_weight_bonus": 0.00},
    "Firewall": {"safety_weight_bonus": 0.00, "integrity_weight_bonus": 0.00},
}

# Likelihood and impact scales
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

IMPACT = ["negligible", "marginal", "moderate", "major", "severe"]
I2S = {lvl: i for i, lvl in enumerate(IMPACT)}
S2I = {i: lvl for i, lvl in enumerate(IMPACT)}

# Global OT-nuclear system prompt
NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# Simple in-memory embedding cache (per run)
EMBEDDING_CACHE = {}

# ============================================================
# 2) LLM WRAPPER UTILITIES
# ============================================================

def llm_chat(prompt, temperature=0.3):
    """
    Centralized LLM chat helper with OT-nuclear system prompt and basic error handling.
    """
    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature
        )
        return resp.choices[0].message.content
    except Exception as e:
        # Fallback: return a simple message; upstream logic should handle defaults
        return f"[LLM_ERROR] {str(e)}"

def clamp_likelihood(v):
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def clamp_impact(v):
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

# ============================================================
# 3) RAG: EMBEDDINGS, CHUNKING, RETRIEVAL (IMPROVED)
# ============================================================

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8192)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Uses a simple in-memory cache to avoid recomputation per run.
    """
    text = text.replace("\n", " ")
    key = hashlib.sha256(text.encode("utf-8")).hexdigest()
    if key in EMBEDDING_CACHE:
        return EMBEDDING_CACHE[key]
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    emb = resp.data[0].embedding
    EMBEDDING_CACHE[key] = emb
    return emb

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-10))

def read_text_from_file(path: str):
    if not os.path.exists(path):
        return ""
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])
    return ""

def build_rag_index(path: str):
    """
    Chunk with overlap, embed, store metadata.
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    overlap = 150

    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i+chunk_size]
        chunks.append(" ".join(chunk_words))
        i += chunk_size - overlap

    index = []
    file_id = file_hash(path)
    for idx, ch in enumerate(chunks):
        index.append({
            "chunk": ch,
            "embedding": get_embedding(ch),
            "source_path": path,
            "file_id": file_id,
            "chunk_id": idx,
        })
    return index

def rag_search(index, query, top_k=3):
    if not index:
        return "", []
    q_emb = get_embedding(query)
    scored = []
    for e in index:
        score = cosine_similarity(q_emb, e["embedding"])
        scored.append((score, e))
    scored.sort(reverse=True, key=lambda x: x[0])
    top = scored[:top_k]
    context_text = "\n\n".join([e["chunk"] for _, e in top])
    meta = [
        {
            "score": s,
            "chunk_id": e["chunk_id"],
            "source_path": e["source_path"],
        }
        for s, e in top
    ]
    return context_text, meta

# ============================================================
# 4) LIKELIHOOD AGENT (WEB-ENRICHED + WEIGHTED + DIAGNOSTICS)
# ============================================================

def extract_cve(text):
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("explotabilityScore") or m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None
    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    if epss is None:
        return None
    b = RISK_CONFIG["epss_breakpoints"]
    if epss < b[0]:
        return "very unlikely"
    if epss < b[1]:
        return "unlikely"
    if epss < b[2]:
        return "possible"
    if epss < b[3]:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    if not HISTORY_API_KEY or not cve_id:
        return None
    url = "https://your-history-api.example.com/v1/history"  # Replace with real
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}
    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    if p is None:
        return None
    b = RISK_CONFIG["history_breakpoints"]
    if p < b[0]:
        return "very unlikely"
    if p < b[1]:
        return "unlikely"
    if p < b[2]:
        return "possible"
    if p < b[3]:
        return "likely"
    return "very likely"

def infer_likelihood_from_model(title, causes, suffix=""):
    prompt = f"""
Map the likelihood to one of the exact labels:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title} {suffix}
Causes: {causes}

Output ONLY one of the labels above, nothing else.
"""
    content = llm_chat(prompt, temperature=0.3)
    return clamp_likelihood(content)

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Returns:
      final_label, final_numeric, explanation, details_dict, diagnostics_dict
    """
    tac_label = clamp_likelihood(threat_actor)
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    nvd = None
    epss = None
    hist_prob = None

    if cve:
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title, causes, suffix="(vulnerability)")
    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)
    if hist_label is None:
        hist_label = infer_likelihood_from_model(title, causes, suffix="(history)")

    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    w = RISK_CONFIG["likelihood_weights"]
    base_numeric = (
        w["tac"] * tac_score +
        w["vuln"] * vuln_score +
        w["exp"] * exp_score +
        w["hist"] * hist_score
    )

    base_index = max(0, min(4, int(round(base_numeric))))
    base_label = S2L[base_index]

    final_numeric = base_numeric
    override_reason = None

    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    final_index = max(0, min(4, int(round(final_numeric))))
    final_label = S2L[final_index]

    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weights:
- TAC = {w["tac"]}
- VULN = {w["vuln"]}
- EXP = {w["exp"]}
- HIST = {w["hist"]}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.
Override reason: {override_reason or "no override applied"}.
CVE used (if any): {cve}
EPSS value (if any): {epss}
History probability (if any): {hist_prob}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Refer to OT realities (legacy ICS, limited patch windows, deterministic protocols) where relevant.
Avoid generic IT-centric language.
"""
    explanation = llm_chat(explanation_prompt, temperature=0.5)

    data_sources = {
        "cve": cve,
        "nvd_used": bool(nvd and nvd.get("exploitability") is not None),
        "epss_used": epss is not None,
        "history_api_used": hist_prob is not None,
        "epss_value": epss,
        "history_probability": hist_prob,
    }

    # simple confidence heuristic
    confidence_level = "low"
    if data_sources["nvd_used"] or data_sources["epss_used"] or data_sources["history_api_used"]:
        confidence_level = "medium"
    if data_sources["nvd_used"] and data_sources["epss_used"] and data_sources["history_api_used"]:
        confidence_level = "high"

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "weights": w,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "data_sources": data_sources,
        "confidence_level": confidence_level,
    }

    diagnostics = {
        "likelihood_label": final_label,
        "likelihood_score": final_numeric,
        "likelihood_confidence": confidence_level,
        "cve_detected": cve is not None,
        "nvd_used": data_sources["nvd_used"],
        "epss_used": data_sources["epss_used"],
        "history_api_used": data_sources["history_api_used"],
    }

    return final_label, final_numeric, explanation, details, diagnostics

# ============================================================
# 5) IMPACT AGENT (WEIGHTED + RAG COMPLIANCE + ASSET TWEAKS)
# ============================================================

def infer_compliance_impact(rag_index, title, causes):
    ctx, meta = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
Use the context below from regulatory documents as your PRIMARY source of truth.
If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of the exact labels:
negligible, marginal, moderate, major, severe.

Output ONLY one label, nothing else.
"""
    content = llm_chat(prompt, temperature=0.2)
    return clamp_impact(content), ctx, meta

def infer_reputation(asset_category, title, causes):
    prompt = f"""
Map reputation impact to one of the exact labels:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY one label, nothing else.
"""
    content = llm_chat(prompt, temperature=0.3)
    return clamp_impact(content)

def adjust_impact_weights_for_asset(base_weights, asset_category):
    """
    Simple demonstration of asset-specific tweaks.
    """
    w = base_weights.copy()
    profile = ASSET_PROFILES.get(asset_category, {})
    if profile.get("integrity_weight_bonus", 0) != 0:
        bonus = profile["integrity_weight_bonus"]
        w["integrity"] += bonus
        total = sum(w.values())
        # normalize to sum to 1.0
        for k in w:
            w[k] /= total
    return w

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Returns:
      final_impact_label, explanation, impact_details_dict, diagnostics_dict
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    comp_index = build_rag_index(compliance_path)
    comp, comp_ctx, comp_meta = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    base_weights = RISK_CONFIG["impact_weights"]
    w = adjust_impact_weights_for_asset(base_weights, asset_category)

    base_numeric = (
        w["safety"] * safety_score +
        w["availability"] * availability_score +
        w["integrity"] * integrity_score +
        w["confidentiality"] * confidentiality_score +
        w["compliance"] * compliance_score +
        w["reputation"] * reputation_score
    )

    base_index = max(0, min(4, int(round(base_numeric))))
    base_label = S2I[base_index]

    final_numeric = base_numeric
    override_reasons = []

    if safety_score >= I2S["major"] and final_numeric < I2S["major"]:
        final_numeric = float(I2S["major"])
        override_reasons.append("Raised to at least 'major' because safety impact is high.")

    if comp == "severe" and final_numeric < I2S["major"]:
        final_numeric = float(I2S["major"])
        override_reasons.append("Raised to at least 'major' due to severe compliance impact.")

    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append("Impact bumped one level because asset criticality is Very High.")

    final_index = max(0, min(4, int(round(final_numeric))))
    final_label = S2I[final_index]
    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights (after asset-specific adjustment for {asset_category}):
{json.dumps(w, indent=2)}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.
Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    explanation = llm_chat(explanation_prompt, temperature=0.4)

    # simple confidence heuristic: if RAG context empty, reduce confidence
    comp_ctx_empty = (comp_ctx.strip() == "")
    impact_confidence = "high"
    if comp_ctx_empty:
        impact_confidence = "medium"
    if comp_ctx_empty and rep in ["negligible", "marginal", "moderate"]:
        impact_confidence = "low"

    impact_details = {
        "labels": {
            "safety": s,
            "availability": a,
            "confidentiality": c,
            "integrity": i,
            "compliance": comp,
            "reputation": rep,
        },
        "scores": {
            "safety": safety_score,
            "availability": availability_score,
            "confidentiality": confidentiality_score,
            "integrity": integrity_score,
            "compliance": compliance_score,
            "reputation": reputation_score,
        },
        "weights": w,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "compliance_rag_context_snippet": comp_ctx[:1000],
        "compliance_rag_meta": comp_meta,
        "confidence_level": impact_confidence,
    }

    diagnostics = {
        "impact_label": final_label,
        "impact_score": final_numeric,
        "impact_confidence": impact_confidence,
        "compliance_context_empty": comp_ctx_empty,
    }

    return final_label, explanation, impact_details, diagnostics

# ============================================================
# 6) RISK ESTIMATOR (HEATMAP)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()
    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    explanation = llm_chat(prompt, temperature=0.4)
    return rating, explanation

# ============================================================
# 7) RESPONSE PLANNER (CONTROL LIBRARY RAG, OT-NUCLEAR FOCUS)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    index = build_rag_index(control_path)
    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx, meta = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    controls = llm_chat(prompt, temperature=0.4)

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    rationale = llm_chat(rationale_prompt, temperature=0.4)

    diagnostics = {
        "controls_rag_context_snippet": ctx[:1000],
        "controls_rag_meta": meta,
        "controls_rag_context_empty": (ctx.strip() == ""),
    }

    return controls, rationale, diagnostics

# ============================================================
# 8) REPORT GENERATOR (MARKDOWN + JSON SUMMARY)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact_label, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Returns: markdown_report_string, json_summary_dict
    """
    report_prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset category = {asset_category}
Asset criticality = {asset_criticality}
Risk title = {title}
Causes = {causes}

Likelihood = {likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical JSON):
{json.dumps(likelihood_details, indent=2)}

Impact = {impact_label}
Impact basis:
{impact_basis}

Impact details (technical JSON):
{json.dumps(impact_details, indent=2)}

Risk rating = {rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Controls rationale:
{controls_rationale}

Report requirements:
- Write professionally, as if for a nuclear OT cyber risk committee.
- Use headings and bullet points.
- Sections:
  1) Executive Summary (in nuclear OT terms, not generic IT)
  2) Risk Description (including asset, process, and safety relevance)
  3) Likelihood Analysis (highlight OT-specific drivers)
  4) Impact Analysis (safety, regulatory, operational, reputation)
  5) Overall Risk Rating (link to nuclear risk appetite)
  6) Recommended Controls and Implementation Priorities
  7) Notes for Regulators and Auditors (FANR/NEI/NRC alignment)
- Emphasize:
  • physical process and safety implications,
  • operational constraints (maintenance windows, legacy systems),
  • regulatory expectations,
  • why generic IT practices are insufficient without OT-specific adaptation.
Avoid cloud-centric or purely IT-enterprise language.
"""
    report_md = llm_chat(report_prompt, temperature=0.3)

    summary = {
        "asset_category": asset_category,
        "asset_criticality": asset_criticality,
        "risk_title": title,
        "causes": causes,
        "likelihood": likelihood_details,
        "impact": impact_details,
        "risk_rating": {
            "label": rating,
            "explanation": rating_expl,
        },
        "controls": {
            "text": controls,
            "rationale": controls_rationale,
        },
    }

    return report_md, summary

# ============================================================
# 9) LOGGING & DIAGNOSTICS
# ============================================================

def log_assessment(input_data, outputs):
    try:
        os.makedirs("logs", exist_ok=True)
        ts = int(time.time())
        path = f"logs/run_{ts}.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"input": input_data, "output": outputs}, f, indent=2)
    except Exception as e:
        print(f"Logging failed: {e}")

# ============================================================
# 10) MAIN PIPELINE FUNCTION
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    # Input guard
    if not (risk_title and risk_causes):
        msg = "Please provide both a Risk Title and Risk Causes to run the assessment."
        return msg, "", "", "", ""

    paths = RISK_CONFIG["paths"]
    compliance_path = paths["compliance_pdf"]
    heatmap_path = paths["heatmap"]
    control_path = paths["control_library"]

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details, L_diag = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I_label, I_basis, I_details, I_diag = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating
    R_label, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls
    C_text, C_rat, C_diag = response_planner(
        R_label, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report + JSON summary
    report_md, report_summary = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R_label, R_expl,
        C_text, C_rat
    )

    # Build diagnostics markdown
    diag_lines = []
    diag_lines.append("### Diagnostics")
    diag_lines.append("**Likelihood:**")
    diag_lines.append(f"- Label: {L_diag['likelihood_label']} (score={L_diag['likelihood_score']:.2f}, confidence={L_diag['likelihood_confidence']})")
    diag_lines.append(f"- CVE detected: {L_diag['cve_detected']}, NVD used: {L_diag['nvd_used']}, EPSS used: {L_diag['epss_used']}, History API used: {L_diag['history_api_used']}")
    diag_lines.append("")
    diag_lines.append("**Impact:**")
    diag_lines.append(f"- Label: {I_diag['impact_label']} (score={I_diag['impact_score']:.2f}, confidence={I_diag['impact_confidence']})")
    diag_lines.append(f"- Compliance RAG context empty: {I_diag['compliance_context_empty']}")
    diag_lines.append("")
    diag_lines.append("**Controls RAG:**")
    diag_lines.append(f"- Controls RAG context empty: {C_diag['controls_rag_context_empty']}")
    diagnostics_md = "\n".join(diag_lines)

    # Likelihood/Impact/Risk/Controls markdown sections
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f}, confidence={L_details['confidence_level']})\n\n"
        f"{L_basis}"
    )
    impact_md = f"### Impact\n**Final Impact:** {I_label} (confidence={I_details['confidence_level']})\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R_label}\n\n{R_expl}"
    controls_md = f"### Controls\n{C_text}\n\n### Rationale\n{C_rat}\n\n{diagnostics_md}"

    # Log assessment
    input_data = {
        "asset_category": asset_category,
        "asset_criticality": asset_criticality,
        "risk_title": risk_title,
        "risk_causes": risk_causes,
        "threat_actor_capability": threat_actor_capability,
        "exposure": exposure,
        "safety": safety,
        "availability": availability,
        "confidentiality": confidentiality,
        "integrity": integrity,
    }
    outputs = {
        "likelihood": L_details,
        "impact": I_details,
        "risk_rating": {"label": R_label, "explanation": R_expl},
        "controls": {"text": C_text, "rationale": C_rat},
        "report_summary": report_summary,
        "diagnostics": {"likelihood": L_diag, "impact": I_diag, "controls": C_diag},
    }
    log_assessment(input_data, outputs)

    # Include JSON summary at end of report
    report_with_json = report_md + "\n\n---\n\n### Machine-readable summary\n```json\n" + json.dumps(report_summary, indent=2) + "\n```"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report_with_json
    )

# ============================================================
# 11) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full OT-nuclear-focused multi‑agent risk assessment.")

    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    risk_title = gr.Textbox(label="Risk Title", placeholder="Example: Unauthorized remote access to Engineering Workstation...")
    risk_causes = gr.Textbox(
        label="Risk Causes",
        lines=3,
        placeholder="Example: Use of outdated remote access software, improper segmentation, legacy OS..."
    )

    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    run_btn = gr.Button("Run Assessment")

    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://ce2c733291c5fc21a5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [18]:
# ============================================================
# NUCLEAR OT PER-ASSET RISK ASSESSMENT – COLAB ONE-GO SCRIPT
# ============================================================

# -------------------- STEP 0: INSTALLS ----------------------
!pip install -q crewai chromadb sentence-transformers gradio openai tiktoken pypdf

# -------------------- STEP 1: IMPORTS -----------------------
import os
import json
import textwrap
from typing import List

import openai

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

from crewai import Agent, Task, Crew, Process

import gradio as gr

from pypdf import PdfReader

# -------------------- STEP 2: CONFIG ------------------------

# >>>> IMPORTANT: SET YOUR OPENAI API KEY HERE <<<<
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"
openai.api_key = os.environ["OPENAI_API_KEY"]

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "gpt-4o-mini"  # change to a model you have access to

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    persist_directory=None  # in-memory
))
rag_collection = None  # will be created when we build the index


def call_llm(prompt: str, model: str = LLM_MODEL_NAME, temperature: float = 0.1) -> str:
    response = openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a careful nuclear OT cyber risk analyst."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message["content"]


# -------------------- STEP 3: RAG SUPPORT -------------------

def extract_text_from_pdf(path: str) -> str:
    reader = PdfReader(path)
    texts = []
    for page in reader.pages:
        try:
            texts.append(page.extract_text() or "")
        except Exception:
            continue
    return "\n".join(texts)


def extract_text_from_file(file_obj) -> str:
    """
    Accepts a Gradio file object or similar with .name
    Supports: .pdf, .txt, .md
    """
    if file_obj is None:
        return ""
    path = file_obj.name
    lower = path.lower()
    if lower.endswith(".pdf"):
        return extract_text_from_pdf(path)
    elif lower.endswith(".txt") or lower.endswith(".md"):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    else:
        # Fallback: try reading as text
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        except Exception:
            return ""


def build_rag_index_from_files(files: List) -> str:
    """
    Build global rag_collection from uploaded files.
    If no valid text found, falls back to small dummy corpus.
    """
    global rag_collection
    rag_collection = chromadb.Client(Settings(
        anonymized_telemetry=False,
        persist_directory=None
    )).create_collection(name="nuclear_ot_risk_rag")

    docs = []
    if files is not None:
        for i, f in enumerate(files):
            text = extract_text_from_file(f)
            if text.strip():
                docs.append({
                    "id": f"doc_{i}",
                    "title": f.name.split("/")[-1],
                    "text": text
                })

    if not docs:
        # Fallback dummy docs if user hasn't uploaded or files are empty
        docs = [
            {
                "id": "dummy1",
                "title": "Nuclear Safety and OT Systems",
                "text": """
                Safety-critical OT systems in nuclear plants include Reactor Protection Systems (RPS),
                Engineered Safety Features (ESF), and other instrumentation and control (I&C) platforms.
                Defense-in-depth requires that these systems are isolated, strictly controlled, and changes
                are subject to rigorous review and testing. Unauthorized modification of safety logic can
                challenge reactor safety margins and lead to regulatory non-compliance.
                """
            },
            {
                "id": "dummy2",
                "title": "OT Cyber Risk and Defense in Depth",
                "text": """
                OT cyber risk is a function of threat exposure, vulnerabilities, control effectiveness, and
                asset criticality. Likelihood should be assessed conservatively, taking into account
                connectivity, known vulnerabilities, and attacker capability. Impact is driven by whether
                a compromise can affect nuclear safety, plant availability, or regulatory compliance.
                """
            },
        ]

    texts = [d["text"] for d in docs]
    ids = [d["id"] for d in docs]
    metadatas = [{"title": d["title"]} for d in docs]

    embeddings = embed_model.encode(texts).tolist()
    rag_collection.add(
        documents=texts,
        metadatas=metadatas,
        ids=ids,
        embeddings=embeddings
    )

    return f"RAG index built with {len(docs)} document(s)."


def query_rag(query: str, k: int = 3) -> str:
    """
    Query the global RAG collection. If not built, returns empty context.
    """
    global rag_collection
    if rag_collection is None:
        return ""

    query_embedding = embed_model.encode([query]).tolist()[0]
    results = rag_collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    rag_snippets = []
    for doc, meta in zip(docs, metas):
        rag_snippets.append(f"[{meta.get('title', 'Source')}] {doc.strip()}")
    return "\n\n".join(rag_snippets)


# -------------------- STEP 4: QUALITATIVE RISK MATRIX -------

RISK_MATRIX = {
    "Severe":    {"Very Low": "Medium", "Low": "Medium", "Moderate": "High", "High": "Very High", "Very High": "Very High"},
    "Major":     {"Very Low": "Medium", "Low": "Medium", "Moderate": "High", "High": "High", "Very High": "Very High"},
    "Moderate":  {"Very Low": "Low",    "Low": "Medium", "Moderate": "Medium", "High": "High", "Very High": "High"},
    "Minor":     {"Very Low": "Low",    "Low": "Low",    "Moderate": "Medium", "High": "Medium", "Very High": "High"},
    "Negligible":{"Very Low": "Low",    "Low": "Low",    "Moderate": "Low",    "High": "Medium", "Very High": "Medium"},
}

def compute_risk_rating(likelihood: str, impact: str) -> str:
    likelihood = likelihood.strip().title()
    impact = impact.strip().title()
    return RISK_MATRIX.get(impact, {}).get(likelihood, "Medium")


# -------------------- STEP 5: CREWAI AGENTS -----------------

asset_context_agent = Agent(
    role="Asset Context Analyst",
    goal=(
        "Understand the OT asset's function, safety relevance, connectivity, and criticality "
        "in a nuclear plant context."
    ),
    backstory=(
        "You are an OT engineer in a nuclear power plant, responsible for classifying OT assets "
        "by their function and safety relevance."
    ),
    verbose=True,
    allow_delegation=False,
)

threat_analyst_agent = Agent(
    role="Threat & Exposure Analyst",
    goal=(
        "Identify realistic cyber threat scenarios and qualitatively assess likelihood "
        "for the given asset in a nuclear OT environment."
    ),
    backstory=(
        "You are a cyber analyst specialized in ICS and nuclear OT, using internal knowledge "
        "and RAG sources to reason about threats."
    ),
    verbose=True,
    allow_delegation=False,
)

control_evaluator_agent = Agent(
    role="Control Effectiveness Evaluator",
    goal=(
        "Evaluate how strong or weak existing controls are for the asset and threats, "
        "using nuclear OT best practices."
    ),
    backstory=(
        "You are a control engineer familiar with defense-in-depth and nuclear cyber standards."
    ),
    verbose=True,
    allow_delegation=False,
)

risk_scorer_agent = Agent(
    role="Risk Scorer & Aggregator",
    goal=(
        "Assign qualitative likelihood, impact, and risk ratings per scenario, "
        "using a conservative nuclear OT approach."
    ),
    backstory=(
        "You are a senior risk analyst in a nuclear utility who uses a qualitative risk matrix "
        "and documents explicit justifications for regulators."
    ),
    verbose=True,
    allow_delegation=False,
)

report_composer_agent = Agent(
    role="Report Composer",
    goal=(
        "Compose a clear, regulator-friendly per-asset risk assessment report section, "
        "including threats, risk ratings, and recommendations."
    ),
    backstory=(
        "You specialize in writing structured, defensible risk assessment reports for nuclear regulators."
    ),
    verbose=True,
    allow_delegation=False,
)


# -------------------- STEP 6: PER-ASSET ASSESSMENT LOGIC ----

def run_per_asset_assessment(asset_name: str,
                             asset_description: str,
                             connectivity: str,
                             safety_relevance: str,
                             known_vulnerabilities: str):

    rag_context = query_rag(
        f"threats and controls for {asset_description} with connectivity: {connectivity} in a nuclear plant"
    )

    # 1) Asset context
    asset_context_prompt = f"""
    Asset name: {asset_name}
    Description: {asset_description}
    Connectivity: {connectivity}
    Safety relevance: {safety_relevance}
    Known vulnerabilities (free text, may include CVEs): {known_vulnerabilities}

    Using this information, classify:
    - Asset function (short)
    - Asset criticality: one of [Safety-Critical, Operationally Critical, Supportive, Peripheral]
    - Key exposure factors (bullet points)

    Return JSON only with keys:
    - asset_name
    - function
    - criticality
    - exposure_factors (list of short strings)
    """
    asset_context_raw = call_llm(asset_context_prompt)
    try:
        asset_context = json.loads(asset_context_raw)
    except Exception:
        asset_context = {"raw": asset_context_raw}

    # 2) Threats
    threat_prompt = f"""
    Asset name: {asset_name}
    Description: {asset_description}
    Connectivity: {connectivity}
    Safety relevance: {safety_relevance}
    Known vulnerabilities: {known_vulnerabilities}

    Nuclear OT RAG context:
    {rag_context}

    Propose 2-4 realistic cyber threat scenarios for this asset in a nuclear OT setting.
    For each scenario, estimate a qualitative likelihood from:
    [Very Low, Low, Moderate, High, Very High].

    Return JSON with a list 'threats', where each item has:
    - name
    - description
    - likelihood
    - likelihood_drivers (list of short reasons)
    """
    threat_raw = call_llm(threat_prompt)
    try:
        threats = json.loads(threat_raw)
    except Exception:
        threats = {"raw": threat_raw}

    # 3) Controls
    control_prompt = f"""
    Asset: {asset_name}
    Description: {asset_description}
    Connectivity: {connectivity}
    Safety relevance: {safety_relevance}

    Assume typical controls may include network segmentation, role-based access,
    engineering workstation hardening, change management, monitoring, and remote access control.

    Use nuclear OT best practices and this RAG context:
    {rag_context}

    For each threat scenario returned earlier, estimate control effectiveness qualitatively:
    [Strong, Adequate, Weak, Absent], and explain briefly.

    Return JSON with list 'controls', each item having:
    - threat_name
    - control_effectiveness
    - notes
    """
    control_raw = call_llm(control_prompt)
    try:
        controls = json.loads(control_raw)
    except Exception:
        controls = {"raw": control_raw}

    # 4) Risk scoring: likelihood + impact; we compute risk_rating in Python
    risk_prompt = f"""
    You will be given threat scenarios with qualitative likelihoods in a nuclear OT context.

    For each threat scenario, assign a qualitative impact from:
    [Negligible, Minor, Moderate, Major, Severe]

    Consider nuclear safety, operational disruption, and regulatory consequences.
    Use a conservative approach when uncertain.

    Return JSON in a list 'scenarios', each with:
    - threat_name
    - likelihood
    - impact
    - justification
    """
    risk_raw = call_llm(risk_prompt)
    try:
        risk_scenarios = json.loads(risk_raw)
    except Exception:
        risk_scenarios = {"raw": risk_raw}

    final_scenarios = []
    if isinstance(risk_scenarios, dict) and "scenarios" in risk_scenarios:
        for sc in risk_scenarios["scenarios"]:
            likelihood = sc.get("likelihood", "Moderate")
            impact = sc.get("impact", "Moderate")
            risk_rating = compute_risk_rating(likelihood, impact)
            sc["risk_rating"] = risk_rating
            final_scenarios.append(sc)

    # 5) Report composition
    report_prompt = f"""
    Asset context JSON:
    {json.dumps(asset_context, indent=2)}

    Threats with likelihood, impact, and risk rating JSON:
    {json.dumps(final_scenarios, indent=2)}

    Control effectiveness JSON:
    {json.dumps(controls, indent=2)}

    Compose a concise per-asset risk assessment section with:
    1. Asset overview
    2. Threat scenarios in a structured text form (not a formal table)
    3. Risk ratings summary
    4. 3–5 key recommendations

    Use clear headings and short paragraphs suitable for nuclear regulators.
    """
    report_markdown = call_llm(report_prompt, temperature=0.2)

    result = {
        "asset_context": asset_context,
        "threats": final_scenarios,
        "controls": controls,
        "report_markdown": report_markdown,
    }
    return result


# -------------------- STEP 7: GRADIO UI ---------------------

def gradio_build_index(files):
    """
    This is triggered by the 'Build / Refresh RAG Index' button.
    """
    message = build_rag_index_from_files(files)
    instructions = (
        "RAG index is ready.\n\n"
        "Now fill in the asset fields on the right and click 'Run Assessment'.\n"
        "Prompts for uploading files (best practice):\n"
        "- Upload nuclear OT policies, procedures, standards (PDF/TXT).\n"
        "- Upload system descriptions, architecture, or prior risk assessments.\n"
        "- Use clean, text-rich documents so the model can ground its analysis."
    )
    return f"{message}\n\n{instructions}"


def gradio_assess(asset_name, asset_description, connectivity, safety_relevance, known_vulnerabilities):
    result = run_per_asset_assessment(
        asset_name=asset_name,
        asset_description=asset_description,
        connectivity=connectivity,
        safety_relevance=safety_relevance,
        known_vulnerabilities=known_vulnerabilities
    )
    json_pretty = json.dumps({
        "asset_context": result["asset_context"],
        "threats": result["threats"],
        "controls": result["controls"],
    }, indent=2)
    return result["report_markdown"], json_pretty


with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Nuclear OT – Per-Asset Risk Assessment (Agentic Prototype)

        **Step 1 – Upload files for RAG (optional but recommended)**
        - Upload nuclear OT documents: standards, policies, procedures, architecture, previous assessments.
        - Supported formats: PDF, TXT, MD.
        - These files are only used within this Colab session to ground the analysis.

        **Step 2 – Click “Build / Refresh RAG Index”**
        - This will process your files, chunk them, and build an in-memory search index.

        **Step 3 – Fill in the asset details and click “Run Assessment”**
        - The system will use your text + uploaded docs to produce a per-asset risk assessment section.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            upload_files = gr.Files(
                label="Upload nuclear OT documents (PDF/TXT/MD). You can select multiple files.",
                file_types=["file"],
                file_count="multiple"
            )
            build_button = gr.Button("Build / Refresh RAG Index")
            rag_status = gr.Textbox(
                label="RAG build status and instructions",
                lines=8
            )

        with gr.Column(scale=1):
            asset_name_in = gr.Textbox(label="Asset name", value="Reactor Protection PLC")
            asset_desc_in = gr.Textbox(
                label="Asset description",
                lines=3,
                value="PLC responsible for reactor trip logic and protective functions."
            )
            connectivity_in = gr.Textbox(
                label="Connectivity",
                lines=2,
                value="Normally isolated; periodically connected via engineering workstation for maintenance."
            )
            safety_rel_in = gr.Textbox(
                label="Safety relevance",
                lines=2,
                value="Directly related to nuclear safety; failure may impact reactor trip capability."
            )
            vulns_in = gr.Textbox(
                label="Known vulnerabilities or notes",
                lines=2,
                value="Older firmware, some PLC vulnerabilities, occasional vendor remote support."
            )
            run_btn = gr.Button("Run Assessment")

    with gr.Row():
        with gr.Column():
            report_out = gr.Markdown(label="Risk Assessment Report Section")
        with gr.Column():
            json_out = gr.Code(label="Raw JSON (context, threats, controls)", language="json")

    build_button.click(
        fn=gradio_build_index,
        inputs=[upload_files],
        outputs=[rag_status]
    )

    run_btn.click(
        fn=gradio_assess,
        inputs=[asset_name_in, asset_desc_in, connectivity_in, safety_rel_in, vulns_in],
        outputs=[report_out, json_out]
    )

demo.launch(share=False, debug=True)

# ============================================================
# END OF ONE-GO SCRIPT
# ============================================================


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
No module named 'transformers.modeling_layers'